Plotting the spectural radius (max eigenvalue) of the digonal blocks of the flux jacobian 
- it should appear a shock structure at mid chord

In [1]:
import numpy as np

# pyau3d
from pyau3d.utils import PltFileUtils, GrpFileUtils, UnkFileUtils
from pyau3d.pv.loader.au3d import arrays2vtk

# matplotlib
import matplotlib.pyplot as plt
from matplotlib.tri import Triangulation
import pandas as pd

# scipy
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import eigs
from scipy.io import mmwrite, mmread
from scipy.spatial import KDTree # for calculating closet point to x,y point
from scipy.sparse import save_npz, load_npz

import time
import pyvista as pv

# %matplotlib widget

In [2]:
# utility functions

def specific_energy(rho, p, ux, uy, uz, gamma=1.4):
    return p / ((gamma - 1.0) * rho) + 0.5 * (ux**2 + uy**2 + uz**2)

def conservative_variables(rst, GAMMA):
    """Return conservative variables U on the surface."""
    E = specific_energy(rst.rho, rst.p, rst.ux, rst.uy, rst.uz, GAMMA)

    U = np.column_stack((
        rst.rho,
        rst.rho * rst.ux,
        rst.rho * rst.uy,
        rst.rho * rst.uz,
        rst.rho * E
    ))
    return U

In [3]:
# build_boundary_list function

# Area normal function - helper function

def area_normals(coord, ifac3=None, ifac4=None):
    """
    Compute area-weighted normal vectors (Ax, Ay, Az) at each nodes,
    accumulated from triangle and/or quadrilateral faces. No PolyData is
    built; only the (N, 3) array is returned.
 
    For each face, the area-normal vector is:
        triangle : 0.5 * cross(v1 - v0, v2 - v0)
        quad     : sum of the two triangle area-normals from splitting
                   the quad (0,1,2) + (0,2,3)
    Each face's area-normal is split equally among its vertices and summed.

    That is how it accounted from the node-centered formulation
 
    Args:
        coord : (N, 3) array of mesh point coordinates
        ifac3 : (M3, 3) array of triangle connectivity, zero-based (or None)
        ifac4 : (M4, 4) array of quad connectivity, zero-based (or None)
 
    Returns:
        anor : (N, 3) array of area-weighted normals (Ax, Ay, Az) per point
    """
    coord = np.asarray(coord, dtype=float)
    anor = np.zeros_like(coord)
 
    if ifac3 is not None and len(ifac3) > 0:
        ifac3 = np.asarray(ifac3)
        v0, v1, v2 = coord[ifac3[:, 0]], coord[ifac3[:, 1]], coord[ifac3[:, 2]]
        face_anor = 0.5 * np.cross(v1 - v0, v2 - v0)   # (M3, 3)
        contrib = -face_anor / 3.0                      # sign matches ref code
        for k in range(3):
            np.add.at(anor, ifac3[:, k], contrib)
 
    if ifac4 is not None and len(ifac4) > 0:
        ifac4 = np.asarray(ifac4)
        v0, v1, v2, v3 = (coord[ifac4[:, 0]], coord[ifac4[:, 1]],
                          coord[ifac4[:, 2]], coord[ifac4[:, 3]])
        n1 = 0.5 * np.cross(v1 - v0, v2 - v0)
        n2 = 0.5 * np.cross(v2 - v0, v3 - v0)
        face_anor = n1 + n2                              # (M4, 3)
        contrib = -face_anor / 4.0
        for k in range(4):
            np.add.at(anor, ifac4[:, k], contrib)
 
    return anor


def boundary_geometry(pltfile, coord, fortfile, flag):
    """
    Wall-normal distance for each boundary node on a given surface.

    ds_i = mean over interior neighbours j of | n_hat_i . (x_j - x_i) |

    Returns
    -------
    (Nb, 5) array: [node_id (0-based), Abx, Aby, Abz, ds]

    """
    # use extract_surface_real to ifac4 as well 
    surface_nodes, tri_connect, quad_connect = pltfile.extract_surface_real(flag = flag)

    # get surface coordinates (0 based indexing)
    surface_coord = coord[surface_nodes]

    surface_area_normals = area_normals(surface_coord, ifac3=tri_connect, ifac4 = quad_connect)

    n_hat = surface_area_normals / np.linalg.norm(surface_area_normals, axis=1, keepdims=True)

    # global node id -> row in surface_nodes / n_hat
    local_index = -np.ones(coord.shape[0], dtype=int)
    local_index[surface_nodes] = np.arange(len(surface_nodes))

    surface_mask = np.zeros(coord.shape[0], dtype=bool)
    surface_mask[surface_nodes] = True

    ds_lists = {}   # boundary node id -> list of projected distances to its interior neighbours

    for row in fortfile:
        a = int(row[0]) - 1   # 0-based
        b = int(row[1]) - 1

        a_surf = surface_mask[a]
        b_surf = surface_mask[b]

        if a_surf and not b_surf:
            node, neigh = a, b
        elif b_surf and not a_surf:
            node, neigh = b, a
        else:
            continue   # both on surface (tangential edge) or both interior -> not what we want

        normal = n_hat[local_index[node]]
        vec = coord[neigh] - coord[node]
        ds_proj = abs(np.dot(vec, normal))

        ds_lists.setdefault(node, []).append(ds_proj)

    node_ids = np.array(sorted(ds_lists.keys()))
    ds = np.array([np.mean(ds_lists[n]) for n in node_ids])

    return np.column_stack([node_ids, surface_area_normals, ds])


def build_boundary_list(pltfile, coord, fortfile, flags_bc_map):
    """
    boundary_list : (B, 12) ndarray
        col 0    : node i (1-based)
        col 1-3  : [Ax, Ay, Az]  outward area-weighted normal
        col 4    : ds
        col 5    : bc_type   0=riemann | 1=slip | 2=noslip
        col 6-8  : u_b, v_b, w_b
        col 9    : T_b
        col 10   : P_b
        col 11   : flag (surface id, for filtering/debugging)
    """
    rows = []
    for flag, bc in flags_bc_map.items():
        geom = boundary_geometry(pltfile, coord, fortfile, flag)
        n = geom.shape[0]
        bc_cols = np.tile([
            float(bc['bc_type']), float(bc['u_b']), float(bc['v_b']),
            float(bc['w_b']), float(bc['T_b']), float(bc['P_b']),
        ], (n, 1))
        node_1based = geom[:, [0]] + 1.0
        flag_col = np.full((n, 1), flag, dtype=float)
        rows.append(np.hstack([node_1based, geom[:, 1:5], bc_cols, flag_col]))
        
    return np.vstack(rows)

In [37]:
# 1. Read neccessary files
Mesh = 21228
Re   = 60
Mach = 0.2
gamma = 1.4
R_gas = 287.0          # confirm units match the solver
mesh_ver = 1
case_name = "OAT"

# dir = f"C:/Users/User/Git/flux_jacobian/cases/v{mesh_ver}_mesh/cylinder_{Mesh}_Re{Re}_M{Mach}"
# dir = f"/home/ahf25/CFD_2d_cylinder_all/Steady/Ma{Mach}/v{mesh_ver}_mesh/2d_cylinder_{Mesh}_Re{Re}" 
# dir = f"/home/ahf25/OAT15/OAT15_M0.73_A35" 

dir = "C:/Users/User/Git/flux_jacobian/cases/OAT15/OAT15_M0.73_A35"
pltfile = PltFileUtils(f"{dir}/{case_name}.plt")
rstfile = UnkFileUtils(f"{dir}/{case_name}.unk", extend=False)  # both rst and unk are fine

fortfile = pd.read_csv(f"{dir}/fort.864", sep=r'\s+', header=None).to_numpy()

rstfile._primitive()
U_list = conservative_variables(rstfile, gamma)
coord  = pltfile.coord

u_in  = 68.0525;  v_in  = 0.0;  w_in  = 0.0
T_in  = 288.15;   P_in  = 1.32702
u_out = 68.0525;  v_out = 0.0;  w_out = 0.0
T_out = 288.15;   P_out = 1.32702

# bc_type   0=riemann | 1=slip | 2=noslip
# flags_bc_map = {
#     1: {'bc_type': 0, 'u_b': u_in,  'v_b': v_in,  'w_b': w_in,  'T_b': T_in,  'P_b': P_in},
#     2: {'bc_type': 0, 'u_b': u_out, 'v_b': v_out, 'w_b': w_out, 'T_b': T_out, 'P_b': P_out},
#     3: {'bc_type': 1, 'u_b': 0.0,   'v_b': 0.0,   'w_b': 0.0,   'T_b': T_in,  'P_b': P_in},
#     4: {'bc_type': 1, 'u_b': 0.0,   'v_b': 0.0,   'w_b': 0.0,   'T_b': T_in,  'P_b': P_in},
#     5: {'bc_type': 1, 'u_b': 0.0,   'v_b': 0.0,   'w_b': 0.0,   'T_b': T_in,  'P_b': P_in},
#     6: {'bc_type': 1, 'u_b': 0.0,   'v_b': 0.0,   'w_b': 0.0,   'T_b': T_in,  'P_b': P_in},
#     7: {'bc_type': 2, 'u_b': 0.0,   'v_b': 0.0,   'w_b': 0.0,   'T_b': T_in,  'P_b': P_in},
# }



# for OAT15 M=0.73, AoA = 18569

u_in  = 252.98;  v_in  = 0.0;  w_in  = 15.47
T_in  = 300;   P_in  = 18569
u_out = 252.98;  v_out = 0.0;  w_out = 15.47
T_out = 300;   P_out = 18569

# bc_type   0=riemann | 1=slip | 2=noslip
flags_bc_map = {
    1: {'bc_type': 0, 'u_b': u_in,  'v_b': v_in,  'w_b': w_in,  'T_b': T_in,  'P_b': P_in},
    2: {'bc_type': 0, 'u_b': u_out, 'v_b': v_out, 'w_b': w_out, 'T_b': T_out, 'P_b': P_out},
    3: {'bc_type': 2, 'u_b': 0.0,   'v_b': 0.0,   'w_b': 0.0,   'T_b': T_in,  'P_b': P_in},
    4: {'bc_type': 1, 'u_b': 0.0,   'v_b': 0.0,   'w_b': 0.0,   'T_b': T_in,  'P_b': P_in},
    5: {'bc_type': 1, 'u_b': 0.0,   'v_b': 0.0,   'w_b': 0.0,   'T_b': T_in,  'P_b': P_in},
}


boundary_list = build_boundary_list(pltfile, coord, fortfile, flags_bc_map)


# # Read the jacobians from ./data
# data_dir = f"/home/ahf25/git/flux_jacobian/data/flux_jacobian_assembly_v4/v{mesh_ver}_mesh"
# data_dir = "C:/Users/User/Git/flux_jacobian/data/flux_jacobian_assembly_v4"
# npz_file = f"jacobian_cylinder_{Mesh}_Re{Re}_M{Mach}_fd.npz"

# # data_dir = f"/home/ahf25/git/flux_jacobian/data/flux_jacobian_assembly_v5"
data_dir = "C:/Users/User/Git/flux_jacobian/data/flux_jacobian_assembly_v5"
npz_file = "jacobian_OAT15_M0.73_A35_fd.npz"

global_flux_jacobian_fd = load_npz(f"{data_dir}/{npz_file}")

J = global_flux_jacobian_fd


In [38]:
# transform PLT file to VTK 
mesh = arrays2vtk(pltfile)

# extract block in multiblock
domain = pv.wrap(mesh.GetBlock(0))    # "Domain 1"  →  pv.MultiBlock
block  = pv.wrap(domain.GetBlock(0))  # "Volume"    →  pv.UnstructuredGrid

In [41]:
N   = J.shape[0] // 5
rows, cols = J.nonzero()
mask       = (rows // 5) == (cols // 5)

diag_rows = rows[mask]
diag_cols = cols[mask]
diag_vals = np.asarray(J[diag_rows, diag_cols]).flatten()

node_idx  = diag_rows // 5
k_idx     = diag_rows % 5
l_idx     = diag_cols % 5

blocks = np.zeros((N, 5, 5))
blocks[node_idx, k_idx, l_idx] = diag_vals

# locate bad nodes
finite_mask = np.isfinite(blocks).all(axis=(1, 2))
bad_nodes   = np.where(~finite_mask)[0]
print(f"Bad nodes: {len(bad_nodes)} / {N}")

# compute eigenvalues only for finite blocks
diag_sr = np.full(N, np.nan)
diag_sr[finite_mask] = np.max(
    np.abs(np.linalg.eigvals(blocks[finite_mask]).real), axis=1)

Bad nodes: 4 / 540156


In [43]:
log_sr = np.log10(diag_sr + 1e-30)          # nan stays nan for bad nodes

block.point_data['log_sr'] = log_sr

pl = pv.Plotter()
pl.add_mesh(block,
            scalars='log_sr',
            cmap='RdBu',
            nan_color='grey',                # bad nodes shown in grey iwth nan_color argument
            show_edges=False,
            clim=[np.nanmin(log_sr), np.nanmax(log_sr)],
            scalar_bar_args={'title': 'log₁₀(spectral radius)'})

if len(bad_nodes) > 0:
    pl.add_mesh(pv.PolyData(coord[bad_nodes]),
                color='yellow', point_size=12,
                render_points_as_spheres=True,
                label='NaN nodes')
    pl.add_legend()

pl.view_xy()
pl.camera.parallel_projection = True
pl.camera.parallel_scale = 0.8
pl.show()

Widget(value='<iframe src="http://localhost:62580/index.html?ui=P_0x1edb93bf250_17&reconnect=auto" class="pyvi…